# First Principles: Numerical Spectrum Algorithms

## 1. Phenomenon

In nature, engineering, and data science, linear transformations define how states evolve, stretch, or resonate.
Analytical eigenvalue computation relies on the roots of the characteristic polynomial $p(\lambda) = \det(A - \lambda I) = 0$.
However, for a matrix $A \in \mathbb{C}^{n \times n}$ where $n \ge 5$, Abel-Ruffini's
theorem proves no general algebraic solution using radicals exists.

Furthermore, numerical root-finding for higher-degree characteristic
polynomials is highly ill-conditioned (e.g., Wilkinson's polynomial).
Thus, discovering the eigenvalues of large matrices must be treated as an iterative numerical phenomenon,
where sequences of vectors or matrices asymptotically align with invariant subspaces.

## 2. Goal

The primary goal is to reliably, efficiently, and accurately compute
the eigenvalues $\lambda_i$ and eigenvectors $v_i$ of a matrix $A$.
Depending on the application context, the goal may be finding only the dominant eigenvalue (largest magnitude),
a subset of eigenvalues (extremal or closest to a target shift), or the entire spectrum (all $n$ eigenvalues).

## 3. Assumptions

1. $A \in \mathbb{R}^{n \times n}$ or $\mathbb{C}^{n \times n}$ is diagonalizable for basic power iteration analyses,
   though Schur decomposition applies universally to all square matrices.

2. For power iteration methods, there exists a unique strictly dominant
   eigenvalue $|\lambda_1| > |\lambda_2| \ge \dots \ge |\lambda_n|$.

3. Floating-point arithmetic introduces roundoff errors, necessitating
   numerical stability analysis (backward stability) and orthogonalization.

## 4. Variables

- $x_k$: The state vector approximation at iteration $k$.
- $A_k$: The matrix state at iteration $k$ in the QR algorithm.
- $Q_k, R_k$: Orthogonal (or unitary) and upper triangular factors at iteration $k$.
- $\mu_k$: The shift parameter used to accelerate iterative convergence.
- $H_m, T_m$: Upper Hessenberg and symmetric tridiagonal matrices in Krylov subspace projections.

## 5. Parameters

- $A$: The input matrix whose spectrum is sought.
- $n$: The dimension of the matrix space.
- $\operatorname{tol}$: The tolerance threshold for convergence (e.g., $10^{-12}$).
- $\text{max\_iter}$: The maximum number of allowed iterations.

## 6. Units and Dimensions

- $A$: Operator mapping $\mathbb{C}^n \to \mathbb{C}^n$ (or $\mathbb{R}^n \to \mathbb{R}^n$).
- $\lambda$: Has the same units as the operator $A$.
- $v, x_k$: State vectors in $\mathbb{C}^n$ (often normalized to unit norm, hence dimensionless).
- Flop counts: Dimensionless operation counts.

## 7. Domain Constraints

- $n \ge 1$, often $n \sim 10^3$ for dense methods, and $n \sim 10^6+$ for sparse methods.
- Matrix $A - \mu I$ must be non-singular for inverse iteration.
- Orthogonal matrices must maintain $Q^T Q = I$ within floating-point precision.

## 8. Governing Principles

- **Invariant Subspaces:** Iterative multiplication $A^k x$ aligns $x$ with the dominant eigenvector subspace.

- **Unitary Similarity Transformations:** Transformations $Q^{\ast} A Q$ preserve eigenvalues and matrix condition numbers perfectly.

- **Krylov Subspace Projection:** Extracting optimal eigenvalue approximations
  from the span $\mathcal{K}_m(A, b) = \operatorname{span}\{b, Ab, A^2b, \dots, A^{m-1}b\}$.

## 9. Mathematical Formulation

The eigenvalue problem is $A v = \lambda v$ with $v \neq 0$.

Numerical algorithms rely on the **Schur Decomposition Theorem**:
For any square matrix $A \in \mathbb{C}^{n \times n}$, there exists a unitary matrix $Q \in \mathbb{C}^{n \times n}$
($Q^{\ast} Q = I$) and an upper triangular matrix $T \in \mathbb{C}^{n \times n}$ such that:

$$
A = Q T Q^{\ast} \quad \Longleftrightarrow \quad Q^{\ast} A Q = T
$$

The diagonal entries of $T$, $t_{ii} = \lambda_i$, are the eigenvalues of $A$.

For a real symmetric matrix $A \in \mathbb{R}^{n \times n}$ (or complex Hermitian matrix $A = A^{\ast}$),
the Schur form collapses to the **Spectral Theorem**:
$Q$ is real orthogonal ($Q^T Q = I$)
and $T = \Lambda = \operatorname{diag}(\lambda_1, \dots, \lambda_n)$ is real diagonal.

Result:

$$
\boxed{A = Q T Q^{\ast}}
$$

## 10. Dominant Eigenvalue Solvers: Power Iteration

**Power Iteration** computes the dominant eigenvalue $\lambda_1$ and its corresponding eigenvector $v_1$.

**Algorithm:**

1. Choose a random initial vector $x_0$ with $\|x_0\|_2 = 1$.

2. For $k = 1, 2, \dots$:

$$
\begin{aligned}
y_k &= A x_{k-1} \\
x_k &= \frac{y_k}{\|y_k\|_2} \\
\lambda^{(k)} &= x_k^{\ast} A x_k \quad (\text{Rayleigh Quotient})
\end{aligned}
$$

**Convergence Derivation:**

1. Expand $x_0$ in the eigenbasis $\{v_1, v_2, \dots, v_n\}$ as $x_0 = \sum_{i=1}^n c_i v_i$ with $c_1 \neq 0$.

2. Apply $A^k$ to $x_0$:

$$
A^k x_0 = c_1 \lambda_1^k v_1 + \sum_{i=2}^n c_i \lambda_i^k v_i = \lambda_1^k \left( c_1 v_1 + \sum_{i=2}^n c_i \left(\frac{\lambda_i}{\lambda_1}\right)^k v_i \right)
$$

3. Because $|\lambda_1| > |\lambda_2| \ge \dots \ge |\lambda_n|$, as $k \to \infty$,
   the summation term decays at the linear rate $\mathcal{O}\left( \left\vert \frac{\lambda_2}{\lambda_1} \right\vert^k \right)$.

Result:

$$
\boxed{\|x_k - (\text{phase}) v_1\|_2 = \mathcal{O}\left( \left\vert \frac{\lambda_2}{\lambda_1} \right\vert^k \right)}
$$

**Derivation of Quadratic Rayleigh Quotient Error for Hermitian Matrices:**

1. Let $x_k = v_1 + \epsilon e$, where $\|v_1\|_2 = 1$, $e \perp v_1$, $\|e\|_2 = 1$,
   and $\epsilon = \mathcal{O}\left( \left\vert \frac{\lambda_2}{\lambda_1} \right\vert^k \right)$.

2. For a Hermitian matrix $A = A^{\ast}$, evaluate the Rayleigh quotient:

$$
\begin{aligned}
\lambda^{(k)} &= \frac{(v_1 + \epsilon e)^{\ast} A (v_1 + \epsilon e)}{(v_1 + \epsilon e)^{\ast} (v_1 + \epsilon e)} \\
&= \frac{\lambda_1 + \epsilon^2 e^{\ast} A e}{1 + \epsilon^2} \\
&= (\lambda_1 + \epsilon^2 e^{\ast} A e)\left(1 - \epsilon^2 + \mathcal{O}(\epsilon^4)\right) \\
&= \lambda_1 + \epsilon^2 (e^{\ast} A e - \lambda_1) + \mathcal{O}(\epsilon^4)
\end{aligned}
$$

3. Thus, for Hermitian matrices, the Rayleigh quotient error doubles the convergence exponent:


Result:

$$
\boxed{|\lambda^{(k)} - \lambda_1| = \mathcal{O}\left( \left\vert \frac{\lambda_2}{\lambda_1} \right\vert^{2k} \right)}
$$

## 11. Inverse Power Iteration and Rayleigh Quotient Iteration

**Inverse Power Iteration:**

Applying Power Iteration to $(A - \mu I)^{-1}$ yields eigenvalues $(\lambda_i - \mu)^{-1}$.
Convergence is driven by the eigenvalue closest to shift $\mu$:

Result:

$$
\boxed{\text{Convergence Rate} = \mathcal{O}\left( \left\vert \frac{\lambda_{\text{closest}} - \mu}{\lambda_{\text{second closest}} - \mu} \right\vert^k \right)}
$$

**Rayleigh Quotient Iteration (RQI):**

Update the shift dynamically at each step: $\mu_k = \frac{x_k^{\ast} A x_k}{x_k^{\ast} x_k}$.

**Proof (Cubic Convergence for Real Symmetric / Hermitian Matrices):**

1. Let $x_k = v + \epsilon e$, where $v$ is an exact unit eigenvector
   ($A v = \lambda v$, $\|v\|_2 = 1$) and $e \perp v$ with $\|e\|_2 = 1$.

2. The Rayleigh quotient shift error is quadratic in $\|\epsilon\|$:

$$
\begin{aligned}
\mu_k - \lambda &= \frac{(v + \epsilon e)^{\ast} A (v + \epsilon e)}{(v + \epsilon e)^{\ast} (v + \epsilon e)} - \lambda \\
&= \epsilon^2 (e^{\ast} A e - \lambda) + \mathcal{O}(\epsilon^4) \\
&= \mathcal{O}(\|\epsilon\|^2)
\end{aligned}
$$

3. In the inverse iteration step, $(A - \mu_k I) y_{k+1} = x_k$.
   
   Since $(A - \mu_k I) v = (\lambda - \mu_k) v$, solving for $y_{k+1}$ yields:

$$
y_{k+1} = (A - \mu_k I)^{-1} (v + \epsilon e) = \frac{1}{\lambda - \mu_k} v + \epsilon (A - \mu_k I)^{-1} e
$$

   Because $\lambda - \mu_k = \mathcal{O}(\|\epsilon\|^2)$, the coefficient of $v$ is $\mathcal{O}(\|\epsilon\|^{-2})$.
   Assuming $\mu_k$ is bounded away from other eigenvalues, $(A - \mu_k I)^{-1} e = \mathcal{O}(1)$.

4. Normalizing $x_{k+1} = \frac{y_{k+1}}{\|y_{k+1}\|_2}$, the ratio of the
   orthogonal error component to the eigenvector component becomes:

$$
\frac{\mathcal{O}(\|\epsilon\|)}{\mathcal{O}(\|\epsilon\|^{-2})} = \mathcal{O}(\|\epsilon\|^3)
$$

Result:

$$
\boxed{\|x_{k+1} - v\|_2 = \mathcal{O}\left(\|x_k - v\|_2^3\right)}
$$

## 12. Jacobi Eigenvalue Algorithm

For symmetric matrices $A = A^T$, the **Jacobi Eigenvalue Algorithm** computes all eigenvalues
by applying a sequence of orthogonal similarity transformations (Givens rotations)
to systematically zero out off-diagonal elements.

**Algorithm:**

1. Find the largest off-diagonal element in magnitude: $|a_{pq}| = \max_{i \neq j} |a_{ij}|$.

2. Form a Givens rotation matrix $J(p, q, \theta)$ which is the identity matrix except for:
   $J_{pp} = \cos \theta, J_{qq} = \cos \theta, J_{pq} = \sin \theta, J_{qp} = -\sin \theta$.

3. Choose $\theta$ to zero out the $(p, q)$ and $(q, p)$ entries in $A_{k+1} = J^T A_k J$. The angle satisfies:

$$
\cot(2\theta) = \frac{a_{qq} - a_{pp}}{2a_{pq}}
$$

4. Repeat until the off-diagonal norm $\|A - \operatorname{diag}(A)\|_F < \operatorname{tol}$.

**Convergence and Complexity:**

Each rotation strictly decreases the sum of squares of the off-diagonal elements.
The algorithm enjoys **quadratic convergence** once the matrix is sufficiently diagonally dominant.
A single full "sweep" (eliminating all $\frac{n(n-1)}{2}$ off-diagonal elements once) requires $\mathcal{O}(n^3)$ operations.

Result:

$$
\boxed{\text{Off-diagonal norm decreases strictly: } \|A_{k+1}^{\text{off}}\|_F^2 = \|A_k^{\text{off}}\|_F^2 - 2a_{pq}^2}
$$

## 13. Full Spectrum Solvers: Standard QR Algorithm

The **Standard QR Algorithm** is the foundational iterative method for dense matrices.

**Algorithm:**

1. Initialize $A_0 = A$.

2. For $k = 0, 1, 2, \dots$:

$$
\begin{aligned}
Q_k R_k &= A_k \quad (\text{QR Factorization}) \\
A_{k+1} &= R_k Q_k
\end{aligned}
$$

**Proof of Similarity and Connection to Subspace Iteration:**

1. Since $A_k = Q_k R_k$ and $Q_k$ is unitary ($Q_k^{\ast} Q_k = I$),
   multiplying by $Q_k^{\ast}$ on the left yields $R_k = Q_k^{\ast} A_k$.
   
   Substituting $R_k$ into $A_{k+1}$:

$$
A_{k+1} = R_k Q_k = (Q_k^{\ast} A_k) Q_k = Q_k^{\ast} A_k Q_k
$$

   Thus, $A_{k+1}$ is unitarily similar to $A_k$.

2. Define the accumulated products $\underline{Q}_k = Q_0 Q_1 \dots Q_{k-1}$ and $\underline{R}_k = R_{k-1} \dots R_1 R_0$.
   
   By induction:

$$
A^k = \underline{Q}_k \underline{R}_k \quad \text{and} \quad A_k = \underline{Q}_k^{\ast} A \underline{Q}_k
$$

   This reveals that the QR algorithm implicitly performs **Simultaneous Subspace Iteration**
   on all columns of the identity matrix.

**Proof of Convergence (Subdiagonal Decay):**

1. Consider the column spaces of $A^k = \underline{Q}_k \underline{R}_k$.
   
   Since $\underline{R}_k$ is upper triangular and non-singular (assuming $A$ is non-singular),
   the first $j$ columns of $A^k$ span the same subspace as the first $j$ columns of $\underline{Q}_k$.

2. Let $A = X \Lambda X^{-1}$ be diagonalizable with eigenvalues $|\lambda_1| > |\lambda_2| > \dots > |\lambda_n|$.
   
   The matrix $A^k = X \Lambda^k X^{-1}$ aligns its dominant column spaces
   with the dominant invariant subspaces of $A$.

3. Specifically, the space spanned by the first $j$ columns of $\underline{Q}_k$
   converges to the space spanned by the first $j$ eigenvectors of $A$.
   
   Because $A_k = \underline{Q}_k^{\ast} A \underline{Q}_k$, the lower-left block of $A_k$ below the $j$-th diagonal
   measures the projection of $A$ applied to the first $j$ Schur vectors
   onto the orthogonal complement of that space.

4. Since the first $j$ Schur vectors converge to an invariant subspace, this projection converges to zero.
   
   The convergence rate of the subdiagonal entry $(A_k)_{j+1, j}$
   is governed by the ratio of the adjacent eigenvalues:

$$
|(A_k)_{j+1, j}| = \mathcal{O}\left( \left\vert \frac{\lambda_{j+1}}{\lambda_j} \right\vert^k \right)
$$

5. As $k \to \infty$, all subdiagonal elements converge to zero, meaning $A_k$ converges
   to an upper triangular matrix (the Schur form), revealing all eigenvalues along its main diagonal.

Result:

$$
\boxed{A^k = \underline{Q}_k \underline{R}_k \quad \text{and} \quad A_k = \underline{Q}_k^{\ast} A \underline{Q}_k}
$$

## 14. Advanced QR: Hessenberg Reduction and Shifts

A naive QR iteration step on a dense matrix costs $\mathcal{O}(n^3)$ flops,
requiring $\mathcal{O}(n^4)$ flops overall.
We optimize this workflow:

1. **Hessenberg Preprocessing:** Reduce $A$ to upper Hessenberg form $H$
   ($h_{ij}=0$ for $i > j+1$) using Householder reflections.
   
   This costs $\frac{10}{3} n^3$ flops once (or $\frac{4}{3} n^3$ for symmetric matrices to tridiagonal form).

2. QR steps on upper Hessenberg matrices preserve Hessenberg structure
   and take only $\mathcal{O}(n^2)$ flops per iteration ($\mathcal{O}(n)$ for symmetric tridiagonal).

3. **Shifted QR Algorithm:**

$$
\begin{aligned}
Q_k R_k &= A_k - \mu_k I \\
A_{k+1} &= R_k Q_k + \mu_k I = Q_k^{\ast} A_k Q_k
\end{aligned}
$$

   Selecting $\mu_k = (A_k)_{n,n}$ (Rayleigh quotient shift) accelerates convergence,
   but it can fail to break symmetry in matrices like $\begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$.
   
   To prevent this failure, we define the **Wilkinson Shift** mathematically:
   $\mu$ is chosen as the eigenvalue of the bottom-right $2 \times 2$ submatrix
   of $A_k$ that is closest to $(A_k)_{n,n}$.
   
   Let the bottom-right $2 \times 2$ submatrix be:

$$
\begin{pmatrix} t_{n-1,n-1} & t_{n-1,n} \\ t_{n,n-1} & t_{nn} \end{pmatrix}
$$

   Define $d = (t_{n-1,n-1} - t_{nn})/2$.
   
   The Wilkinson shift formula is:

$$
\mu = t_{nn} - \frac{t_{n,n-1}^2}{d + \operatorname{sgn}(d)\sqrt{d^2 + t_{n,n-1}^2}}
$$

   This shift breaks the stagnation on matrices with symmetric off-diagonals and zero diagonals,
   guaranteeing at least quadratic (and often cubic) convergence for the subdiagonal elements.

4. **Implicit QR and the Implicit Q Theorem:**
   
   The **Implicit Q Theorem** states that if $Q^{\ast} A Q = H$ and $V^{\ast} A V = G$ are unreduced
   upper Hessenberg matrices and $Q e_1 = V e_1$, then $V = Q D$ where
   $D = \operatorname{diag}(\pm 1, \dots, \pm 1)$ for the real case
   (for complex matrices, $D$ is a diagonal unitary matrix with $|d_{ii}| = 1$).
   
   Using "Bulge Chasing", an implicit QR step applies a single Householder transformation
   matching $(A - \mu I) e_1$ to the first column of $A$ and chases the resulting
   non-Hessenberg bulge down the subdiagonals in $\mathcal{O}(n^2)$ flops, achieving
   the exact result of a shifted QR step without explicitly computing $A_k - \mu_k I$.

Result:

$$
\boxed{Q e_1 = V e_1 \implies V = Q D \quad \text{(Implicit Q Theorem)}}
$$

## 15. Divide-and-Conquer Eigen Solver

For symmetric tridiagonal matrices (resulting from symmetric Hessenberg reduction),
the Divide-and-Conquer algorithm is the fastest method
($\mathcal{O}(n^{2.3})$ on average, $\mathcal{O}(n^2)$ in practice).

**Mechanism and Derivation of the Secular Equation:**

1. Decompose the tridiagonal matrix $T \in \mathbb{R}^{n \times n}$ at index $k$ into subproblems:

$$
T = \begin{bmatrix} T_1 & 0 \\ 0 & T_2 \end{bmatrix} + \beta u u^T
$$

   where $T_1 \in \mathbb{R}^{k \times k}$, $T_2 \in \mathbb{R}^{(n-k) \times (n-k)}$ are tridiagonal,
   $\beta = t_{k+1, k}$ is the subdiagonal splitting element, and $u = e_k + e_{k+1} \in \mathbb{R}^n$.

   > [!NOTE]
   > The submatrices $T_1$ and $T_2$ are not simply the principal submatrices of $T$.
   >
   > They are modified via rank-1 tearing: $T_1 = T(1{:}k, 1{:}k) - \beta e_k e_k^T$
   > and $T_2 = T(k{+}1{:}n, k{+}1{:}n) - \beta e_1 e_1^T$.
   >

2. Recursively solve the subproblems $T_1 = Q_1 D_1 Q_1^T$ and $T_2 = Q_2 D_2 Q_2^T$.
   
   Block-diagonalizing gives:

$$
T = \begin{bmatrix} Q_1 & 0 \\ 0 & Q_2 \end{bmatrix} \left( \begin{bmatrix} D_1 & 0 \\ 0 & D_2 \end{bmatrix} + \beta z z^T \right) \begin{bmatrix} Q_1^T & 0 \\ 0 & Q_2^T \end{bmatrix}
$$

   where $z = \begin{bmatrix} Q_1 & 0 \\ 0 & Q_2 \end{bmatrix}^T u = \begin{bmatrix} Q_1^T e_k \\ Q_2^T e_1 \end{bmatrix}$.

3. Let $D = \operatorname{diag}(d_1, \dots, d_n) = \begin{bmatrix} D_1 & 0 \\ 0 & D_2 \end{bmatrix}$.
   
   The eigenvalues of $T$ are the roots of $\det(D + \beta z z^T - \lambda I) = 0$.
   
   Factoring out $D - \lambda I$:

$$
\det(D + \beta z z^T - \lambda I) = \det(D - \lambda I) \det\left(I + \beta (D - \lambda I)^{-1} z z^T\right)
$$

4. Applying the Matrix Determinant Lemma ($\det(I + x y^T) = 1 + y^T x$) yields the **Secular Equation**:

$$
f(\lambda) = 1 + \beta \sum_{i=1}^n \frac{z_i^2}{d_i - \lambda} = 0
$$

Result:

$$
\boxed{f(\lambda) = 1 + \beta \sum_{i=1}^n \frac{z_i^2}{d_i - \lambda} = 0 \quad \text{where } z = \begin{bmatrix} Q_1^T e_k \\ Q_2^T e_1 \end{bmatrix}}
$$

## 16. Large Sparse Spectrum Solvers: Arnoldi and Lanczos

For sparse matrices where $\mathcal{O}(n^3)$ operations are prohibitive, we project $A$
onto the Krylov subspace $\mathcal{K}_m(A, b) = \operatorname{span}\{b, Ab, A^2b, \dots, A^{m-1}b\}$.

**Arnoldi Iteration (General Matrices):**

Uses Gram-Schmidt orthogonalization to build an orthonormal basis $Q_m = [q_1, \dots, q_m]$ satisfying:

$$
A Q_m = Q_m H_m + h_{m+1, m} q_{m+1} e_m^T
$$

The eigenvalues of the small $m \times m$ upper Hessenberg matrix $H_m = Q_m^{\ast} A Q_m$
(Ritz values) approximate the extremal eigenvalues of $A$.

Result:

$$
\boxed{A Q_m = Q_m H_m + h_{m+1, m} q_{m+1} e_m^T}
$$

**Lanczos Iteration (Hermitian / Symmetric Matrices):**

1. If $A = A^{\ast}$, then $H_m^{\ast} = (Q_m^{\ast} A Q_m)^{\ast} = Q_m^{\ast} A^{\ast} Q_m = H_m$.
   
   An upper Hessenberg matrix that is Hermitian must be **symmetric tridiagonal** $T_m$.

2. As a result, entry $h_{ij} = 0$ for $|i - j| > 1$, and the Arnoldi recurrence
   collapses from an $m$-term Gram-Schmidt orthogonalization to a **three-term recurrence**:

$$
A q_j = \beta_{j-1} q_{j-1} + \alpha_j q_j + \beta_j q_{j+1}
$$

   or equivalently:

$$
\beta_j q_{j+1} = A q_j - \alpha_j q_j - \beta_{j-1} q_{j-1}
$$

   where $\alpha_j = q_j^{\ast} A q_j$ and $\beta_j = \|A q_j - \alpha_j q_j - \beta_{j-1} q_{j-1}\|_2$.
   
   This requires storing only three vectors, enabling massive sparse matrix computations.

Result:

$$
\boxed{A = A^{\ast} \implies H_m = T_m \quad \text{(Tridiagonal)} \implies A q_j = \beta_{j-1} q_{j-1} + \alpha_j q_j + \beta_j q_{j+1}}
$$

## 17. Implicitly Restarted Arnoldi Method (IRAM)

**The Problem:**
In standard Arnoldi iteration, the basis $Q_m$ grows by one vector per iteration.
As $m$ becomes large, the $\mathcal{O}(m^2 n)$ orthogonalization cost and memory storage become prohibitive.
Simply restarting the Arnoldi process with a new random vector discards valuable spectral information.

**The Solution (ARPACK/eigs):**
The Implicitly Restarted Arnoldi Method (IRAM) condenses the $m$-step Arnoldi
factorization back to a $k$-step factorization ($k < m$),
preserving the components corresponding to the desired eigenvalues.

This is achieved by applying $p = m - k$ implicit QR steps to the Hessenberg
matrix $H_m$ with specific shifts $\mu_1, \dots, \mu_p$ chosen as the unwanted Ritz values.

This acts as a polynomial filter, damping out the eigencomponents associated with the shifts.

Result:

$$
\boxed{\text{IRAM applies } p \text{ implicit QR steps to filter unwanted Ritz values, reducing an } m\text{-step to a } k\text{-step basis}}
$$

## 18. Golub-Kahan Bidiagonalization

For a general matrix $A \in \mathbb{R}^{m \times n}$, we often need its Singular Value
Decomposition (SVD), which is intimately connected to the eigenvalue problem
of $A^T A$ or $\begin{bmatrix} 0 & A \\ A^T & 0 \end{bmatrix}$.

Starting from $A$, the **Golub-Kahan Bidiagonalization** generates orthonormal matrices
$U$ and $V$ such that $A = U B V^T$, where $B$ is an upper bidiagonal matrix.

The process uses a Lanczos-like alternating recurrence:

$$
\begin{aligned}
\alpha_i u_i &= A v_i - \beta_{i-1} u_{i-1} \\
\beta_i v_{i+1} &= A^T u_i - \alpha_i v_i
\end{aligned}
$$

where $\alpha_i$ and $\beta_i$ are chosen to normalize $u_i$ and $v_{i+1}$.

The singular values of $A$ are exactly the singular values of $B$: $\sigma_i(A) = \sigma_i(B)$.
The bidiagonal matrix $B$ is then typically solved using a specialized SVD QR algorithm.

Result:

$$
\boxed{A V_k = U_k B_k \quad \text{and} \quad A^T U_k = V_{k+1} \tilde{B}_k^T}
$$

## 19. Loss of Orthogonality and Reorthogonalization

**The Problem:**
In exact arithmetic, Lanczos vectors $q_j$ are orthogonal.
In floating-point arithmetic, as Ritz values converge to true eigenvalues, orthogonality is lost rapidly.

**Paige's Theorem (Full Statement):**
C.C. Paige proved mathematically that in finite-precision Lanczos,
the loss of orthogonality is inextricably linked to convergence.
Orthogonality between Lanczos vectors $q_{k+1}$ and Ritz vector $y_i = Q_k s_i$
is lost *precisely* as the Ritz pair $(\theta_i, y_i)$ converges to an eigenpair of $A$.

Specifically, the projection of a new Lanczos vector onto a Ritz vector $y_i$ satisfies:

Result:

$$
\boxed{\|q_{k+1}^T y_i\| \approx \frac{\varepsilon \|A\|_2}{\beta_k \|s_{ki}\|} = \frac{\varepsilon \|A\|_2}{\|A y_i - \theta_i y_i\|_2}}
$$

where $\varepsilon$ is machine precision and $\beta_k \|s_{ki}\|$ is the residual norm of Ritz pair $(\theta_i, y_i)$.
As the Ritz pair converges, the residual norm in the denominator approaches zero,
amplifying roundoff errors and causing $q_{k+1}$ to lose orthogonality in the direction of $y_i$.

**Solutions:**

1. **Full Reorthogonalization:** Gram-Schmidt against all previous vectors (higher computational cost).

2. **Selective Reorthogonalization:** Orthogonalize only against converged Ritz vectors.

3. **Lanczos without Reorthogonalization:** Allow "ghost" (spurious multiple)
   eigenvalues to appear, and filter them out via Cullum-Willoughby heuristics.

## 20. Algorithmic Complexity / Flop Counts

- Power / Inverse Iteration: $\mathcal{O}(n^2)$ per step for dense matrices,
  $\mathcal{O}(\operatorname{nnz}(A))$ for sparse matrices.
- Rayleigh Quotient Iteration: $\mathcal{O}(n^3)$ per step if factorizing $A - \mu_k I$
  from scratch ($\mathcal{O}(n)$ per step if tridiagonal).
- Hessenberg Reduction: $\frac{10}{3} n^3$ flops for general dense matrices
  ($\frac{4}{3} n^3$ for symmetric tridiagonal reduction).
- Shifted QR step on Hessenberg: $\mathcal{O}(n^2)$ flops per iteration
  ($\mathcal{O}(n)$ for symmetric tridiagonal).
- Total Shifted QR Algorithm: $\mathcal{O}(n^3)$ flops to compute all eigenvalues of a dense matrix.
- Arnoldi Iteration: $\mathcal{O}(\operatorname{nnz}(A) \cdot m + m^2 n)$ flops for $m$
  Krylov steps (due to full $m$-term orthogonalization).
- Lanczos Iteration (without reorthogonalization): $\mathcal{O}(\operatorname{nnz}(A) \cdot m + m n)$
  flops for $m$ Krylov steps (due to 3-term recurrence).

## 21. Verification Strategy

1. **Trace and Determinant:** Verify $\sum_{i=1}^n \lambda_i = \operatorname{tr}(A)$ and $\prod_{i=1}^n \lambda_i = \det(A)$.

2. **Residual Norms:** Check that $\| A v_i - \lambda_i v_i \|_2 \le \operatorname{tol} \cdot \|A\|_2$.

3. **Orthogonality:** For symmetric matrices, verify computed eigenvectors satisfy $\|V^T V - I\|_F \approx 0$.

## 22. Interpretation and Real-World Applications

- **Google PageRank:**
  The principal eigenvector of the Google web graph stochastic matrix is computed via Power Iteration.

- **Quantum Mechanics:**
  The Schrödinger eigenvalue problem $H \psi = E \psi$ requires finding energy levels
  of massive sparse Hamiltonian operators using Lanczos algorithms.

- **Structural Dynamics:**
  Natural frequencies of vibration correspond to square roots of stiffness matrix
  eigenvalues; critical low frequencies are computed using Shifted Inverse Iteration.

## 23. Application in AI

- **Principal Component Analysis (PCA):**
  Computing top $k$ eigenvalues/eigenvectors of covariance matrices via
  randomized SVD or Lanczos for dimensionality reduction.

- **Spectral Clustering:**
  Constructing graph Laplacians and solving for the second-smallest eigenvector
  (Fiedler vector) to partition graph data using sparse eigensolvers.

- **Hessian Spectrum in Deep Learning:**
  Analyzing loss landscapes by computing extremal eigenvalues of neural network
  Hessians via matrix-free vector products (Pearlmutter's trick) and Lanczos.